# DhakaRoadNet: Dataset Download, Verification, and YOLOv8 Prep

This notebook prepares the custom Dhaka road-object dataset for the next training stage. It downloads or reuses the Roboflow YOLOv8 export, validates the dataset structure, scans labels for common annotation issues, summarizes class balance, saves research-friendly reports, and prepares a YOLOv8-ready data configuration for the next notebook.

Pipeline covered here:
1. Environment and path setup
2. Roboflow dataset download or reuse
3. Dataset schema and label-quality checks
4. Split and class-distribution reports
5. Reproducible annotation visualizations
6. YOLOv8 training handoff


## 1. Imports and Global Settings

The notebook keeps randomness deterministic so visual QA examples can be reproduced across runs.


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import os
import random

import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from dotenv import find_dotenv, load_dotenv
from IPython.display import display

try:
    from roboflow import Roboflow
except ImportError:
    Roboflow = None

SPLITS = ("train", "valid", "test")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RANDOM_SEED = 42

ROBOFLOW_WORKSPACE = "east-west-university-r3hrx"
ROBOFLOW_PROJECT = "main-road-object-dataset"
ROBOFLOW_VERSION = 4
ROBOFLOW_FORMAT = "yolov8"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 2. Environment Setup and Project Paths

The dataset can be reused locally without a Roboflow API key. The API key is required only when the dataset folder is missing or when `OVERWRITE_DATASET` is set to `True`.


In [ ]:
def resolve_project_root() -> tuple[Path, Path | None]:
    """Find the project root from .env first, then from common repo markers."""
    env_path = find_dotenv(usecwd=True)
    if env_path:
        env_file = Path(env_path).resolve()
        return env_file.parent, env_file

    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".git").exists() or (candidate / "README.md").exists():
            return candidate, None

    return current, None


PROJECT_ROOT, ENV_FILE = resolve_project_root()
DATASET_DIR = PROJECT_ROOT / "data" / "roboflow"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if ENV_FILE and ENV_FILE.exists():
    load_dotenv(ENV_FILE)
else:
    print("No .env file found. Local dataset QA can still run if data/roboflow exists.")

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
OVERWRITE_DATASET = False

print(f"Project root      : {PROJECT_ROOT}")
print(f"Environment file  : {ENV_FILE if ENV_FILE else 'not found'}")
print(f"Dataset directory : {DATASET_DIR}")
print(f"Reports directory : {REPORTS_DIR}")
print(f"Random seed       : {RANDOM_SEED}")


## 3. Download or Reuse the Roboflow Dataset

This cell avoids redundant network calls. It uses the existing local export when `data/roboflow` already contains files.


In [ ]:
def dataset_exists(dataset_dir: Path) -> bool:
    return dataset_dir.exists() and any(dataset_dir.iterdir())


def download_or_reuse_dataset(dataset_dir: Path, overwrite: bool = False) -> Path:
    if dataset_exists(dataset_dir) and not overwrite:
        print("Dataset already exists locally. Skipping Roboflow download.")
        return dataset_dir

    if not ROBOFLOW_API_KEY:
        raise RuntimeError(
            "ROBOFLOW_API_KEY is missing. Add it to .env or place the dataset in data/roboflow."
        )

    if Roboflow is None:
        raise ImportError("roboflow is not installed. Install it before downloading the dataset.")

    print("Downloading dataset from Roboflow...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download(
        ROBOFLOW_FORMAT,
        location=str(dataset_dir),
        overwrite=overwrite,
    )
    return Path(dataset.location)


dataset_path = download_or_reuse_dataset(DATASET_DIR, overwrite=OVERWRITE_DATASET).resolve()
print(f"Dataset ready at: {dataset_path}")


## 4. Load and Validate `data.yaml`

This section confirms the class list and prepares a YOLOv8-compatible local YAML file if the Roboflow path entries do not resolve from the current dataset location.


In [ ]:
def normalize_class_names(names) -> list[str]:
    if isinstance(names, dict):
        return [names[idx] for idx in sorted(names)]
    return list(names)


def load_dataset_yaml(dataset_dir: Path) -> tuple[dict, list[str], Path]:
    yaml_path = dataset_dir / "data.yaml"
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing dataset YAML: {yaml_path}")

    with yaml_path.open("r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    class_names = normalize_class_names(config.get("names", []))
    declared_nc = int(config.get("nc", len(class_names)))
    if declared_nc != len(class_names):
        raise ValueError(
            f"data.yaml declares nc={declared_nc}, but contains {len(class_names)} class names."
        )

    if not class_names:
        raise ValueError("data.yaml does not contain class names.")

    return config, class_names, yaml_path


def split_image_dir_from_yaml(yaml_path: Path, split_value: str) -> Path:
    split_path = Path(split_value)
    if split_path.is_absolute():
        return split_path
    return (yaml_path.parent / split_path).resolve()


def validate_yaml_split_paths(config: dict, yaml_path: Path) -> pd.DataFrame:
    rows = []
    split_keys = {"train": "train", "valid": "val", "test": "test"}
    for split, yaml_key in split_keys.items():
        value = config.get(yaml_key)
        resolved = split_image_dir_from_yaml(yaml_path, value) if value else None
        expected = dataset_path / split / "images"
        rows.append(
            {
                "split": split,
                "yaml_key": yaml_key,
                "yaml_value": value,
                "resolved_path": str(resolved) if resolved else None,
                "resolved_exists": bool(resolved and resolved.exists()),
                "expected_exists": expected.exists(),
                "expected_path": str(expected),
            }
        )
    return pd.DataFrame(rows)


def write_local_yolov8_yaml(dataset_dir: Path, config: dict, class_names: list[str]) -> Path:
    local_config = {
        "path": dataset_dir.resolve().as_posix(),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(class_names),
        "names": class_names,
    }
    if "roboflow" in config:
        local_config["roboflow"] = config["roboflow"]

    local_yaml_path = dataset_dir / "data_yolov8.yaml"
    with local_yaml_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(local_config, f, sort_keys=False, allow_unicode=False)
    return local_yaml_path


data_config, class_names, original_yaml_path = load_dataset_yaml(dataset_path)
yaml_path_report = validate_yaml_split_paths(data_config, original_yaml_path)

needs_local_yaml = not yaml_path_report["resolved_exists"].all()
YOLO_DATA_YAML = write_local_yolov8_yaml(dataset_path, data_config, class_names) if needs_local_yaml else original_yaml_path

print(f"Original data.yaml : {original_yaml_path}")
print(f"YOLOv8 data YAML   : {YOLO_DATA_YAML}")
print(f"Number of classes  : {len(class_names)}")
print("Class names:")
for idx, name in enumerate(class_names):
    print(f"  {idx:02d}: {name}")

print("\nYAML split path check:")
display(yaml_path_report)


## 5. Dataset Health and Label QA

The scan checks image-label pairing, empty labels, malformed YOLO rows, invalid class IDs, and normalized bounding-box coordinates outside `[0, 1]`.


In [ ]:
def list_image_files(image_dir: Path) -> list[Path]:
    if not image_dir.exists():
        return []
    return sorted(
        path for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def list_label_files(label_dir: Path) -> list[Path]:
    if not label_dir.exists():
        return []
    return sorted(label_dir.glob("*.txt"))


def parse_yolo_label_file(label_path: Path, num_classes: int) -> tuple[list[dict], list[dict], bool]:
    records = []
    issues = []
    lines = label_path.read_text(encoding="utf-8").splitlines()
    non_empty_lines = [line for line in lines if line.strip()]
    is_empty = len(non_empty_lines) == 0

    for line_number, line in enumerate(lines, start=1):
        stripped = line.strip()
        if not stripped:
            continue

        parts = stripped.split()
        if len(parts) != 5:
            issues.append(
                {
                    "file": str(label_path),
                    "line": line_number,
                    "issue_type": "malformed_row",
                    "details": f"Expected 5 values, found {len(parts)}",
                }
            )
            continue

        try:
            class_id = int(parts[0])
        except ValueError:
            issues.append(
                {
                    "file": str(label_path),
                    "line": line_number,
                    "issue_type": "invalid_class_id",
                    "details": f"Class id is not an integer: {parts[0]}",
                }
            )
            continue

        try:
            x_center, y_center, width, height = map(float, parts[1:])
        except ValueError:
            issues.append(
                {
                    "file": str(label_path),
                    "line": line_number,
                    "issue_type": "non_numeric_box",
                    "details": stripped,
                }
            )
            continue

        if class_id < 0 or class_id >= num_classes:
            issues.append(
                {
                    "file": str(label_path),
                    "line": line_number,
                    "issue_type": "class_id_out_of_range",
                    "details": f"class_id={class_id}, num_classes={num_classes}",
                }
            )

        coords = (x_center, y_center, width, height)
        if any(value < 0 or value > 1 for value in coords) or width <= 0 or height <= 0:
            issues.append(
                {
                    "file": str(label_path),
                    "line": line_number,
                    "issue_type": "bbox_out_of_range",
                    "details": f"x={x_center}, y={y_center}, w={width}, h={height}",
                }
            )

        records.append(
            {
                "label_file": label_path,
                "class_id": class_id,
                "x_center": x_center,
                "y_center": y_center,
                "bbox_width": width,
                "bbox_height": height,
                "bbox_area": width * height,
            }
        )

    return records, issues, is_empty


def scan_dataset(dataset_dir: Path, class_names: list[str]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    summary_rows = []
    issue_rows = []
    split_class_counts = {}
    split_annotation_records = {}

    for split in SPLITS:
        image_dir = dataset_dir / split / "images"
        label_dir = dataset_dir / split / "labels"
        image_files = list_image_files(image_dir)
        label_files = list_label_files(label_dir)

        image_by_stem = {path.stem: path for path in image_files}
        label_by_stem = {path.stem: path for path in label_files}
        images_without_labels = sorted(set(image_by_stem) - set(label_by_stem))
        labels_without_images = sorted(set(label_by_stem) - set(image_by_stem))

        class_counter = Counter()
        annotation_records = []
        empty_label_stems = []
        bbox_areas = []

        for label_file in label_files:
            records, issues, is_empty = parse_yolo_label_file(label_file, len(class_names))
            issue_rows.extend({**issue, "split": split} for issue in issues)
            if is_empty:
                empty_label_stems.append(label_file.stem)

            for record in records:
                class_counter[record["class_id"]] += 1
                annotation_records.append(record)
                bbox_areas.append(record["bbox_area"])

        for stem in images_without_labels:
            issue_rows.append(
                {
                    "split": split,
                    "file": str(image_by_stem[stem]),
                    "line": None,
                    "issue_type": "image_without_label_file",
                    "details": "Image has no matching .txt label file.",
                }
            )

        for stem in labels_without_images:
            issue_rows.append(
                {
                    "split": split,
                    "file": str(label_by_stem[stem]),
                    "line": None,
                    "issue_type": "label_without_image_file",
                    "details": "Label file has no matching image file.",
                }
            )

        total_images = len(image_files)
        total_boxes = sum(class_counter.values())
        background_images = len(empty_label_stems) + len(images_without_labels)
        split_class_counts[split] = class_counter
        split_annotation_records[split] = annotation_records

        summary_rows.append(
            {
                "split": split,
                "images": total_images,
                "label_files": len(label_files),
                "background_images": background_images,
                "empty_label_files": len(empty_label_stems),
                "images_without_label_file": len(images_without_labels),
                "labels_without_image_file": len(labels_without_images),
                "annotation_boxes": total_boxes,
                "boxes_per_image": round(total_boxes / total_images, 3) if total_images else 0,
                "median_bbox_area_norm": float(np.median(bbox_areas)) if bbox_areas else 0,
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty and summary_df["images"].sum() > 0:
        summary_df["image_split_percent"] = (summary_df["images"] / summary_df["images"].sum() * 100).round(2)

    class_rows = []
    for class_id, class_name in enumerate(class_names):
        row = {"class_id": class_id, "class_name": class_name}
        for split in SPLITS:
            row[split] = split_class_counts.get(split, Counter()).get(class_id, 0)
        row["total"] = sum(row[split] for split in SPLITS)
        class_rows.append(row)

    class_distribution_df = pd.DataFrame(class_rows).sort_values("total", ascending=False).reset_index(drop=True)
    issue_df = pd.DataFrame(issue_rows)
    return summary_df, class_distribution_df, issue_df, split_annotation_records


summary_df, class_distribution_df, issue_df, split_annotation_records = scan_dataset(dataset_path, class_names)

print("Dataset summary:")
display(summary_df)

print("\nClass distribution:")
display(class_distribution_df)

if issue_df.empty:
    print("\nNo malformed labels, invalid boxes, or image-label pairing issues found.")
else:
    print(f"\nFound {len(issue_df)} dataset issue(s). Showing the first 20:")
    display(issue_df.head(20))


## 6. Save Dataset Reports

The CSV outputs make the dataset status easy to cite in reports, presentations, and the next training notebook.


In [ ]:
dataset_summary_path = REPORTS_DIR / "dataset_summary.csv"
class_distribution_path = REPORTS_DIR / "class_distribution.csv"
dataset_issues_path = REPORTS_DIR / "dataset_issues.csv"
yaml_path_check_path = REPORTS_DIR / "yaml_path_check.csv"

yaml_path_report.to_csv(yaml_path_check_path, index=False)
summary_df.to_csv(dataset_summary_path, index=False)
class_distribution_df.to_csv(class_distribution_path, index=False)

issue_columns = ["split", "file", "line", "issue_type", "details"]
if issue_df.empty:
    pd.DataFrame(columns=issue_columns).to_csv(dataset_issues_path, index=False)
else:
    issue_df.to_csv(dataset_issues_path, index=False)

print(f"Saved dataset summary      : {dataset_summary_path}")
print(f"Saved class distribution   : {class_distribution_path}")
print(f"Saved YAML path check      : {yaml_path_check_path}")
if not issue_df.empty:
    print(f"Saved dataset issues       : {dataset_issues_path}")


## 7. Plot Class Distribution Across Splits

The stacked horizontal chart helps identify class imbalance before training with a pretrained YOLOv8 model.


In [ ]:
def plot_class_distribution(class_df: pd.DataFrame, save_path: Path) -> None:
    df_plot = class_df.sort_values("total", ascending=True).copy()
    fig_height = max(8, 0.42 * len(df_plot))
    fig, ax = plt.subplots(figsize=(13, fig_height))

    colors = {
        "train": "#2563eb",
        "valid": "#f59e0b",
        "test": "#10b981",
    }

    left = np.zeros(len(df_plot))
    y_labels = df_plot["class_name"]
    for split in SPLITS:
        values = df_plot[split].to_numpy()
        ax.barh(
            y_labels,
            values,
            left=left,
            label=split,
            color=colors[split],
            edgecolor="white",
            linewidth=0.8,
        )
        left += values

    max_total = max(df_plot["total"].max(), 1)
    for y_index, total in enumerate(df_plot["total"]):
        ax.text(
            total + max_total * 0.01,
            y_index,
            f"{int(total)}",
            va="center",
            ha="left",
            fontsize=9,
            color="#222222",
        )

    ax.set_title("DhakaRoadNet Class Distribution by Split", fontsize=15, fontweight="bold", pad=14)
    ax.set_xlabel("Annotation count")
    ax.set_ylabel("Class")
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(title="Split", loc="lower right")
    plt.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()


class_distribution_fig_path = FIGURES_DIR / "class_distribution.png"
plot_class_distribution(class_distribution_df, class_distribution_fig_path)
print(f"Saved class distribution figure: {class_distribution_fig_path}")


## 8. Visualize Reproducible Annotated Samples

The first figure saves a single annotated sample. The second figure saves a reproducible grid for broader visual inspection.


In [ ]:
def yolo_to_xyxy(record: dict, image_width: int, image_height: int) -> tuple[int, int, int, int]:
    x_center = record["x_center"] * image_width
    y_center = record["y_center"] * image_height
    box_width = record["bbox_width"] * image_width
    box_height = record["bbox_height"] * image_height

    x1 = int(round(x_center - box_width / 2))
    y1 = int(round(y_center - box_height / 2))
    x2 = int(round(x_center + box_width / 2))
    y2 = int(round(y_center + box_height / 2))

    x1 = max(0, min(x1, image_width - 1))
    y1 = max(0, min(y1, image_height - 1))
    x2 = max(0, min(x2, image_width - 1))
    y2 = max(0, min(y2, image_height - 1))
    return x1, y1, x2, y2


def load_rgb_image(image_path: Path) -> np.ndarray | None:
    image = cv2.imread(str(image_path))
    if image is None:
        return None
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def read_valid_records(label_path: Path, num_classes: int) -> list[dict]:
    if not label_path.exists():
        return []
    records, _, _ = parse_yolo_label_file(label_path, num_classes)
    return [record for record in records if 0 <= record["class_id"] < num_classes]


def class_color(class_id: int, num_classes: int):
    cmap = plt.get_cmap("tab20", max(num_classes, 1))
    return cmap(class_id % max(num_classes, 1))


def draw_annotations(ax, image: np.ndarray, label_path: Path, class_names: list[str]) -> int:
    height, width = image.shape[:2]
    records = read_valid_records(label_path, len(class_names))
    ax.imshow(image)
    ax.axis("off")

    for record in records:
        class_id = record["class_id"]
        color = class_color(class_id, len(class_names))
        x1, y1, x2, y2 = yolo_to_xyxy(record, width, height)
        box_width = max(1, x2 - x1)
        box_height = max(1, y2 - y1)

        rect = patches.Rectangle(
            (x1, y1),
            box_width,
            box_height,
            linewidth=1.8,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)

        label = class_names[class_id]
        text_x = x1
        text_y = y1 - 4
        horizontal_alignment = "left"

        if x1 > width * 0.68:
            text_x = x2
            horizontal_alignment = "right"
        if text_y < 8:
            text_y = min(height - 2, y1 + 12)

        ax.text(
            text_x,
            text_y,
            label,
            color="white",
            fontsize=8,
            fontweight="bold",
            ha=horizontal_alignment,
            va="bottom",
            bbox={"facecolor": color, "edgecolor": "none", "alpha": 0.88, "pad": 1.5},
            clip_on=True,
        )

    return len(records)


def select_sample_images(dataset_dir: Path, split: str = "train", count: int = 9, seed: int = RANDOM_SEED) -> list[Path]:
    image_dir = dataset_dir / split / "images"
    label_dir = dataset_dir / split / "labels"
    image_files = list_image_files(image_dir)
    annotated_images = []

    for image_path in image_files:
        label_path = label_dir / f"{image_path.stem}.txt"
        if label_path.exists() and label_path.read_text(encoding="utf-8").strip():
            annotated_images.append(image_path)

    candidates = annotated_images if annotated_images else image_files
    if not candidates:
        return []

    rng = random.Random(seed)
    sample_size = min(count, len(candidates))
    return rng.sample(candidates, sample_size)


def plot_single_sample(image_path: Path, save_path: Path, split: str = "train") -> None:
    label_path = dataset_path / split / "labels" / f"{image_path.stem}.txt"
    image = load_rgb_image(image_path)
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    fig, ax = plt.subplots(figsize=(8, 6))
    box_count = draw_annotations(ax, image, label_path, class_names)
    ax.set_title(f"{split}/{image_path.name} - {box_count} boxes", fontsize=11)
    plt.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()


def plot_sample_grid(image_paths: list[Path], save_path: Path, split: str = "train", columns: int = 3) -> None:
    if not image_paths:
        print("No images available for visualization.")
        return

    rows = int(np.ceil(len(image_paths) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 5, rows * 4))
    axes = np.atleast_1d(axes).reshape(rows, columns)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, image_path in zip(axes.ravel(), image_paths):
        label_path = dataset_path / split / "labels" / f"{image_path.stem}.txt"
        image = load_rgb_image(image_path)
        if image is None:
            ax.set_title(f"Unreadable: {image_path.name}", fontsize=9)
            continue
        box_count = draw_annotations(ax, image, label_path, class_names)
        ax.set_title(f"{image_path.name}\n{box_count} boxes", fontsize=8)

    plt.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()


sample_images = select_sample_images(dataset_path, split="train", count=9, seed=RANDOM_SEED)
print(f"Selected {len(sample_images)} reproducible training sample(s).")
for image_path in sample_images:
    print(f"  - {image_path.name}")

if sample_images:
    single_sample_path = FIGURES_DIR / "sample_visualization.png"
    sample_grid_path = FIGURES_DIR / "sample_grid_visualization.png"
    plot_single_sample(sample_images[0], single_sample_path, split="train")
    plot_sample_grid(sample_images, sample_grid_path, split="train", columns=3)
    print(f"Saved sample visualization     : {single_sample_path}")
    print(f"Saved sample grid visualization: {sample_grid_path}")


## 9. YOLOv8 Training Handoff

Use this section as the bridge to the next notebook. It does not start training; it prints the dataset config and a starter cell for training with a pretrained YOLOv8 model.


In [ ]:
total_images = int(summary_df["images"].sum())
total_boxes = int(summary_df["annotation_boxes"].sum())

print("DhakaRoadNet dataset is ready for YOLOv8 training.")
print(f"Dataset root       : {dataset_path}")
print(f"YOLOv8 data YAML   : {YOLO_DATA_YAML}")
print(f"Total images       : {total_images}")
print(f"Total annotations  : {total_boxes}")
print(f"Classes            : {len(class_names)}")
print(f"Reports saved in   : {REPORTS_DIR}")

starter_cell = f"""
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=r"{YOLO_DATA_YAML}",
    imgsz=640,
    epochs=100,
    batch=16,
    project=r"{PROJECT_ROOT / 'model' / 'runs'}",
    name="yolov8n_dhakaroadnet",
)
""".strip()

print("\nStarter cell for 02_training.ipynb:")
print(starter_cell)
